In [7]:
#Import Visual Libraries
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Import util Libraries
from sklearn.model_selection import train_test_split as ts
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier,BaggingClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.metrics import matthews_corrcoef
from statsmodels.tsa.stattools import acf
from timeseries_features import (
    hurst,
    permutation_entropy,
    kyles_lambda,
    bekker_parkinson_vol,
    amihuds_lambda,
    hasbroucks_lambda,
    corwin_schultz_hl,
    roll_measure,
)
import yfinance as yf
import quantstats as qs
import talib as ta



In [19]:
from blk_utils import cprint

In [8]:
## Download Data

def process_initial_df(raw_df,resample_freq):
    df = raw_df.copy()
    # Resample
    df.resample(resample_freq).agg({
        'Open':'first',
        'High':'max',
        'Low':'min',
        'Close':'last',
        'Volume':'sum'
    })
    # Drop NA
    df.dropna(inplace=True)
    return df

In [23]:
SYMBOL = "OIL.NS" # SPDR Financial ETF

raw_df = yf.Ticker(SYMBOL).history(period="max")
cprint(raw_df)


-------------------------------------------------------------------------------
dataframe information
-------------------------------------------------------------------------------
HEAD num rows: 5
                                Open       High        Low      Close  \
Date                                                                    
2009-09-30 00:00:00+05:30  70.437120  74.338155  70.051521  73.342010   
2009-10-01 00:00:00+05:30  70.822697  75.430676  70.822697  74.958305   
2009-10-05 00:00:00+05:30  74.036089  74.929406  73.046373  73.300224   
2009-10-06 00:00:00+05:30  73.894700  74.370277  72.757166  73.579788   
2009-10-07 00:00:00+05:30  74.151767  74.595211  73.264871  73.367699   

                              Volume  Dividends  Stock Splits  
Date                                                           
2009-09-30 00:00:00+05:30  148110052        0.0           0.0  
2009-10-01 00:00:00+05:30   23056899        0.0           0.0  
2009-10-05 00:00:00+05:30    6898

In [24]:
df = process_initial_df(raw_df,"1D")
cprint(df)

-------------------------------------------------------------------------------
dataframe information
-------------------------------------------------------------------------------
HEAD num rows: 5
                                Open       High        Low      Close  \
Date                                                                    
2009-09-30 00:00:00+05:30  70.437120  74.338155  70.051521  73.342010   
2009-10-01 00:00:00+05:30  70.822697  75.430676  70.822697  74.958305   
2009-10-05 00:00:00+05:30  74.036089  74.929406  73.046373  73.300224   
2009-10-06 00:00:00+05:30  73.894700  74.370277  72.757166  73.579788   
2009-10-07 00:00:00+05:30  74.151767  74.595211  73.264871  73.367699   

                              Volume  Dividends  Stock Splits  
Date                                                           
2009-09-30 00:00:00+05:30  148110052        0.0           0.0  
2009-10-01 00:00:00+05:30   23056899        0.0           0.0  
2009-10-05 00:00:00+05:30    6898

In [26]:
# Split dataset into Training set, Validation Set and Test Set
train_size = int(0.7 * len(df))
val_size = int(0.15 * len(df))
test_size = len(df) - train_size - val_size
train_df = df.iloc[:train_size]
val_df = df.iloc[train_size:train_size+val_size]
test_df = df.iloc[test_size:]

# check dataset
print("Train Size: ", len(train_df))
cprint(train_df)
print("Validation Size: ", len(val_df))
cprint(val_df)
print("Test Size: ", len(test_df))
cprint(test_df)

Train Size:  2566
-------------------------------------------------------------------------------
dataframe information
-------------------------------------------------------------------------------
HEAD num rows: 5
                                Open       High        Low      Close  \
Date                                                                    
2009-09-30 00:00:00+05:30  70.437120  74.338155  70.051521  73.342010   
2009-10-01 00:00:00+05:30  70.822697  75.430676  70.822697  74.958305   
2009-10-05 00:00:00+05:30  74.036089  74.929406  73.046373  73.300224   
2009-10-06 00:00:00+05:30  73.894700  74.370277  72.757166  73.579788   
2009-10-07 00:00:00+05:30  74.151767  74.595211  73.264871  73.367699   

                              Volume  Dividends  Stock Splits  
Date                                                           
2009-09-30 00:00:00+05:30  148110052        0.0           0.0  
2009-10-01 00:00:00+05:30   23056899        0.0           0.0  
2009-10-05 00:0